# Análise interativa — Avaliação Estética Automática (ArtCLIP/APDDv2)

Versão notebook do `scripts/analyze.py`, pra você mexer nas visualizações direto
aqui no Colab sem precisar descrever cada detalhe pro Claude.

**Fluxo de trabalho:**
1. Rode as células de setup (1–3) uma vez.
2. Rode as células de cada experimento — os gráficos aparecem inline, e também
   são salvos em `reports/figures/`.
3. Edite qualquer célula de gráfico/tabela à vontade (cores, títulos, bins,
   layout...) e rode de novo — é só Python + matplotlib puro, nada de mágica.
4. Quando estiver satisfeita, baixe o notebook (**File → Download → .ipynb** ou
   **.py**) e me manda de volta — eu re-sincronizo tudo em `scripts/analyze.py`.

As funções e nomes de variável seguem exatamente o `scripts/analyze.py` atual,
só que quebradas em células por gráfico/tabela em vez de uma função gigante —
assim dá pra rodar e comparar peça por peça.

## 1. Setup

In [ ]:
%matplotlib inline
import json
import os
import random
import warnings
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import (friedmanchisquare, kendalltau, ks_2samp, mannwhitneyu,
                          pearsonr, spearmanr, wasserstein_distance, wilcoxon)
from scipy.special import rel_entr
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
random.seed(42)
np.random.seed(42)
print("OK — libs carregadas.")

## 2. Dados de entrada

Suba o `outputs.zip` que você baixa do cluster (mesmo arquivo que usa localmente
no Windows). Se quiser as comparações com o ground truth humano (`load_human_gt`),
suba também o `APDDv2-10023.csv` quando for pedido — é opcional, o resto do
notebook funciona sem ele.

In [ ]:
from google.colab import files
import zipfile, shutil

DATA_DIR = "/content/data"
os.makedirs(DATA_DIR, exist_ok=True)

print("Selecione o outputs.zip baixado do cluster:")
uploaded = files.upload()

for fname in uploaded:
    if fname.lower().endswith(".zip"):
        with zipfile.ZipFile(fname, "r") as z:
            z.extractall(DATA_DIR)
        print(f"Extraído: {fname} -> {DATA_DIR}")
        os.remove(fname)


def _resolve_outputs_dir(base, folder_name="outputs"):
    """Os zips do cluster costumam vir com uma pasta aninhada (outputs/outputs/)."""
    nested = os.path.join(base, folder_name, folder_name)
    if os.path.isdir(nested):
        return nested
    direct = os.path.join(base, folder_name)
    if os.path.isdir(direct):
        return direct
    return base


OUTPUTS_DIR = _resolve_outputs_dir(DATA_DIR, "outputs")
REPORTS_DIR = os.path.join(DATA_DIR, "reports")
FIG_DIR = os.path.join(REPORTS_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

print("OUTPUTS_DIR:", OUTPUTS_DIR)
print("Experimentos encontrados:", sorted(os.listdir(OUTPUTS_DIR)) if os.path.isdir(OUTPUTS_DIR) else "(nada encontrado)")
print("FIG_DIR (saída):", FIG_DIR)

In [ ]:
# Opcional — ground truth humano (APDDv2-10023.csv), usado por load_human_gt().
# Rode esta célula só se quiser as comparações "Human GT" do Exp1/Exp2; pode pular.
APDDV2_CSV_PATH = None

_ans = input("Quer subir o APDDv2-10023.csv agora? (s/N): ").strip().lower()
if _ans == "s":
    uploaded_csv = files.upload()
    for fname in uploaded_csv:
        if fname.lower().endswith(".csv"):
            APDDV2_CSV_PATH = os.path.join(DATA_DIR, fname)
            shutil.move(fname, APDDV2_CSV_PATH)
    print("APDDV2_CSV_PATH:", APDDV2_CSV_PATH)
else:
    print("Pulado — comparações com Human GT ficarão vazias (o resto funciona normal).")

## 3. Configuração (paleta, hachuras, labels, etc.)

Mesmo conteúdo do `configs/analysis_local.yaml` — só os `paths` que mudam pra
apontar pros dados que você acabou de subir. Pra trocar cor/hachura/label é só
editar o dicionário abaixo e rodar de novo as células dos gráficos.

In [ ]:
cfg = {
    "palette": {
        "original":  "#448FF2",
        "janus_1b":  "#33A650",
        "janus_7b":  "#F2A007",
        "mnist":     "#F23838",
        "highlight": "#1A00F2",
    },
    "hatches": {
        "original":  "",
        "janus_1b":  "///",
        "janus_7b":  "xxx",
        "mnist":     "...",
        "highlight": "---",
    },
    "linestyles": {
        "original": "solid",
        "janus_1b": "dashed",
        "janus_7b": "dotted",
        "mnist":    "dashdot",
    },
    "markers": {
        "original": "o",
        "janus_1b": "s",
        "janus_7b": "^",
        "mnist":    "D",
    },
    "lang": "en",  # troque pra "pt" e rode de novo pra virar os títulos/labels
    "labels": {
        "en": {
            "datasets": {
                "original": "Original", "janus_1b": "Janus-Pro-1B", "janus_7b": "Janus-Pro-7B",
                "portinari": "Portinari", "mnist": "MNIST",
                "exp2a": "AI Captions (2a)", "exp2b": "Human Captions (2b)",
            },
            "attributes": {
                "Total aesthetic score": "Total Score", "Theme and logic": "Theme & Logic",
                "Creativity": "Creativity", "Layout and composition": "Layout & Composition",
                "Space and perspective": "Space & Perspective", "The sense of order": "Sense of Order",
                "Light and shadow": "Light & Shadow", "Color": "Color",
                "Details and texture": "Details & Texture", "The overall": "Overall", "Mood": "Mood",
            },
            "noise_types": {"gaussian": "Gaussian Noise", "blur": "Blur", "shapes": "Geometric Shapes"},
            "axes": {
                "score": "Aesthetic Score", "noise_level": "Noise Level (%)", "degradation": "Degradation (%)",
                "frame_idx": "Frame Index", "category": "Artistic Category",
                "sample_rank": "Sample (ranked by original score)",
            },
            "titles": {
                "dist_scores": "Score Distribution — APDDv2 Baseline", "by_category": "Score by Artistic Category",
                "original_vs_janus": "Original vs. Generated Images",
                "score_diff": "Score Difference (Original − Generated)", "radar": "Mean Attribute Scores",
                "noise_impact": "Noise Impact on Aesthetic Score", "noise_threshold": "Score vs. Noise Level by Type",
                "temporal_consist": "Temporal Score Consistency",
                "degradation_detect": "Progressive Degradation Detection",
                "art_vs_noart": "Art vs. Non-Art Discrimination", "portinari_compare": "Portinari: Original vs. Generated",
                "caption_source": "Caption Source Impact on Generated Score",
            },
        },
        "pt": {
            "datasets": {
                "original": "Original", "janus_1b": "Janus-Pro-1B", "janus_7b": "Janus-Pro-7B",
                "portinari": "Portinari", "mnist": "MNIST",
                "exp2a": "Captions do Janus (2a)", "exp2b": "Captions Humanas (2b)",
            },
            "attributes": {
                "Total aesthetic score": "Score Total", "Theme and logic": "Tema e Lógica",
                "Creativity": "Criatividade", "Layout and composition": "Layout e Composição",
                "Space and perspective": "Espaço e Perspectiva", "The sense of order": "Senso de Ordem",
                "Light and shadow": "Luz e Sombra", "Color": "Cor",
                "Details and texture": "Detalhes e Textura", "The overall": "Geral", "Mood": "Humor",
            },
            "noise_types": {"gaussian": "Ruído Gaussiano", "blur": "Desfoque", "shapes": "Formas Geométricas"},
            "axes": {
                "score": "Score Estético", "noise_level": "Nível de Ruído (%)", "degradation": "Degradação (%)",
                "frame_idx": "Índice de Frame", "category": "Categoria Artística",
                "sample_rank": "Amostra (ordenada pelo score original)",
            },
            "titles": {
                "dist_scores": "Distribuição dos Scores — Baseline APDDv2", "by_category": "Score por Categoria Artística",
                "original_vs_janus": "Originais vs. Imagens Geradas",
                "score_diff": "Diferença de Score (Original − Gerado)", "radar": "Médias por Atributo",
                "noise_impact": "Impacto do Ruído no Score Estético", "noise_threshold": "Score vs. Nível de Ruído por Tipo",
                "temporal_consist": "Consistência Temporal dos Scores",
                "degradation_detect": "Detecção de Degradação Progressiva",
                "art_vs_noart": "Discriminação Arte vs. Não-Arte", "portinari_compare": "Portinari: Originais vs. Geradas",
                "caption_source": "Impacto da Fonte de Caption no Score",
            },
        },
    },
    "figures": {"dpi": 150, "figsize": [12, 6], "figsize_sq": [8, 8], "font_size": 12,
                "title_size": 14, "legend_loc": "best", "grid": True, "format": "png"},
    "stats": {"alpha": 0.05, "tests": ["ttest", "mannwhitney", "anova", "pearson", "spearman"]},
    "clustering": {"n_clusters": 3, "n_top": 10, "method": "kmeans"},
    "temporal": {"n_videos_to_plot": 5},
    "score_attributes": [
        "Total aesthetic score", "Theme and logic", "Creativity", "Layout and composition",
        "Space and perspective", "The sense of order", "Light and shadow", "Color",
        "Details and texture", "The overall", "Mood",
    ],
    "radar_attributes": [
        "Theme and logic", "Creativity", "Layout and composition", "Space and perspective",
        "The sense of order", "Light and shadow", "Color", "Details and texture", "The overall", "Mood",
    ],
    "paths": {
        "outputs": OUTPUTS_DIR,
        "reports": REPORTS_DIR,
        "apddv2_csv": APDDV2_CSV_PATH or "",
        "apddv2_images": "",
    },
}
print("cfg pronto. lang =", cfg["lang"])

## 4. Funções auxiliares — carregamento de dados

In [ ]:
def load_scores(exp_dir, source):
    path = os.path.join(exp_dir, "scores", f"scores_{source}.csv")
    if not os.path.exists(path):
        return None
    return pd.read_csv(path)


def load_pipeline_data(exp_dir):
    path = os.path.join(exp_dir, "pipeline_data.json")
    if not os.path.exists(path):
        return []
    with open(path) as f:
        return json.load(f)


def available_sources(exp_dir):
    scores_dir = os.path.join(exp_dir, "scores")
    if not os.path.isdir(scores_dir):
        return []
    return [f.replace("scores_", "").replace(".csv", "")
            for f in os.listdir(scores_dir) if f.startswith("scores_") and f.endswith(".csv")]


def _stem(filename):
    return os.path.splitext(os.path.basename(str(filename)))[0]


def load_human_gt(cfg):
    """Carrega APDDv2-10023.csv como ground truth humano (opcional)."""
    path = cfg["paths"].get("apddv2_csv", "")
    if not path or not os.path.exists(path):
        return None
    try:
        df = pd.read_csv(path, encoding="ISO-8859-1")
    except Exception:
        try:
            df = pd.read_csv(path)
        except Exception:
            return None
    fn_col = next((c for c in df.columns if "filename" in c.lower()), None)
    if fn_col is None:
        fn_col = df.columns[0]
    df = df.rename(columns={fn_col: "filename"})
    df["stem"] = df["filename"].apply(_stem)
    return df


def _exp1_scores_dir(cfg, strategy="uniform_bins"):
    """exp1_apdd tem 2 pastas (sampling.strategies) — usa uniform_bins como o Exp1 'principal'."""
    base = os.path.join(cfg["paths"]["outputs"], f"exp1_apdd_{strategy}")
    if os.path.isdir(base):
        return base
    return os.path.join(cfg["paths"]["outputs"], "exp1_apdd")


def _available_attrs(cfg, *dfs):
    all_attrs = cfg["score_attributes"]
    dfs_valid = [d for d in dfs if d is not None]
    if not dfs_valid:
        return all_attrs
    return [a for a in all_attrs if all(a in d.columns for d in dfs_valid)]


print("OK — helpers de dados definidos.")

## 5. Funções auxiliares — visual (paleta/hachuras/labels/save)

In [ ]:
def _palette(cfg):
    return cfg["palette"]


def _hatches(cfg):
    return cfg["hatches"]


def _linestyles(cfg):
    return cfg["linestyles"]


def _markers(cfg):
    return cfg["markers"]


SOURCE_KEYS = {
    "original": "original", "Janus-Pro-1B": "janus_1b", "Janus-Pro-7B": "janus_7b",
    "mnist": "mnist", "Human_description": "original", "Gen_description": "janus_1b",
}


def _skey(source_name):
    return SOURCE_KEYS.get(source_name, "original")


def L(cfg, *keys):
    node = cfg["labels"][cfg["lang"]]
    for k in keys:
        node = node[k]
    return node


def attr_label(cfg, attr):
    return cfg["labels"][cfg["lang"]]["attributes"].get(attr, attr)


def ds_label(cfg, key):
    return cfg["labels"][cfg["lang"]]["datasets"].get(key, key)


def save(fig, path, cfg):
    """Versão notebook: salva em disco E mostra inline (plt.show antes de fechar)."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    dpi = cfg.get("figures", {}).get("dpi", 150)
    fmt = cfg.get("figures", {}).get("format", "png")
    fig.savefig(path, dpi=dpi, bbox_inches="tight", format=fmt)
    plt.show()
    plt.close(fig)


def _style_median(bp):
    for med in bp.get("medians", []):
        med.set_color("black")
        med.set_linewidth(2.5)


print("OK — helpers visuais definidos.")

## 6. Estatística (Friedman + Wilcoxon + CLD, diferença de distribuição)

In [ ]:
def _compact_letters(group_names, means, pval_dict, alpha=0.05):
    """Compact Letter Display: grupos sem diferença significativa compartilham uma letra."""
    if len(group_names) < 2:
        return {g: "a" for g in group_names}
    sorted_names = sorted(group_names, key=lambda g: means.get(g, 0), reverse=True)
    letter_groups = []
    for g in sorted_names:
        placed = []
        for idx, lg in enumerate(letter_groups):
            can_join = all(pval_dict.get(tuple(sorted([g, m])), 1.0) >= alpha for m in lg)
            if can_join:
                placed.append(idx)
        if placed:
            for idx in placed:
                letter_groups[idx].add(g)
        else:
            letter_groups.append({g})
    result = {g: "" for g in group_names}
    for idx, lg in enumerate(letter_groups):
        char = chr(ord("a") + idx)
        for g in lg:
            result[g] += char
    return result


def friedman_wilcoxon(groups_dict, attrs, alpha=0.05):
    """groups_dict: {group_name: DataFrame com coluna 'stem' e colunas de atributos}."""
    result = {}
    group_names = list(groups_dict.keys())
    for attr in attrs:
        dfs = []
        for name, df in groups_dict.items():
            if df is None or attr not in df.columns:
                continue
            if "stem" not in df.columns:
                df = df.copy()
                df["stem"] = df["filename"].apply(_stem)
            dfs.append(df[["stem", attr]].rename(columns={attr: name}))
        if len(dfs) < 2:
            continue
        merged = dfs[0]
        for d in dfs[1:]:
            merged = merged.merge(d, on="stem", how="inner")
        merged = merged.dropna()
        if len(merged) < 3:
            continue
        valid = [g for g in group_names if g in merged.columns]
        vals = [merged[g].values for g in valid]
        friedman_p = 1.0
        if len(vals) >= 3:
            try:
                _, friedman_p = friedmanchisquare(*vals)
            except Exception:
                pass
        pval_dict = {}
        for g1, g2 in combinations(valid, 2):
            pair = tuple(sorted([g1, g2]))
            try:
                _, p = wilcoxon(merged[g1].values, merged[g2].values)
                pval_dict[pair] = p
            except Exception:
                pval_dict[pair] = 1.0
        means = {g: float(merged[g].mean()) for g in valid}
        stds = {g: float(merged[g].std()) for g in valid}
        letters = _compact_letters(valid, means, pval_dict, alpha)
        result[attr] = {g: {"mean": means[g], "std": stds[g], "letter": letters[g], "n": len(merged)}
                        for g in valid}
        result[attr]["_friedman_p"] = friedman_p
    return result


def distribution_diff(s1, s2, name1="A", name2="B"):
    a = s1.dropna().values
    b = s2.dropna().values
    if len(a) < 2 or len(b) < 2:
        return None
    ks_stat, ks_p = ks_2samp(a, b)
    w = wasserstein_distance(a, b)
    bins = 50
    r = (min(a.min(), b.min()), max(a.max(), b.max()))
    if r[0] == r[1]:
        kl = 0.0
    else:
        ph, _ = np.histogram(a, bins=bins, range=r, density=True)
        qh, _ = np.histogram(b, bins=bins, range=r, density=True)
        ph += 1e-10; qh += 1e-10
        kl = float(np.sum(rel_entr(ph, qh)))
    return {"pair": (name1, name2), "ks_stat": ks_stat, "ks_p": ks_p,
            "wasserstein": w, "kl": kl, "n1": len(a), "n2": len(b)}


def apply_nan_mask(df_ref, df_target, attr):
    """Alinha ref e target por stem e propaga o NaN do ref pro target (view-only)."""
    if "stem" not in df_ref.columns:
        df_ref = df_ref.copy(); df_ref["stem"] = df_ref["filename"].apply(_stem)
    if "stem" not in df_target.columns:
        df_target = df_target.copy(); df_target["stem"] = df_target["filename"].apply(_stem)
    merged = df_ref[["stem", attr]].merge(
        df_target[["stem", attr]].rename(columns={attr: attr + "_t"}), on="stem", how="inner")
    mask = merged[attr].isna()
    ref_vals = merged[attr].copy()
    tgt_vals = merged[attr + "_t"].copy()
    tgt_vals[mask] = np.nan
    return ref_vals, tgt_vals


print("OK — helpers de estatística definidos.")

## 7. Renderizadores de tabela/gráfico compartilhados

In [ ]:
def render_stat_table_png(fw_result, attrs, group_names, path, cfg, title="", best_is="highest"):
    """Tabela Friedman/Wilcoxon: 'mean ± std^letter'; melhor valor em negrito."""
    col_labels = [ds_label(cfg, _skey(g)) if _skey(g) in cfg["labels"][cfg["lang"]]["datasets"] else g
                  for g in group_names]
    col_labels_full = col_labels + ["Friedman p"]
    row_labels = [attr_label(cfg, a) for a in attrs]
    cell_text = []
    for attr in attrs:
        row_text = []
        row_info = fw_result.get(attr, {})
        fp = row_info.get("_friedman_p", None)
        means = {g: row_info[g]["mean"] for g in group_names if g in row_info}
        best_g = (max(means, key=means.get) if best_is == "highest" else min(means, key=means.get)) if means else None
        for g in group_names:
            if g not in row_info:
                row_text.append("—")
            else:
                m, s, l = row_info[g]["mean"], row_info[g]["std"], row_info[g]["letter"]
                cell = f"{m:.2f}±{s:.2f}{l}"
                if g == best_g:
                    cell = f"*{cell}*"
                row_text.append(cell)
        row_text.append(f"{fp:.3f}" if (fp is not None and fp >= 0.001) else ("<0.001" if fp is not None else "—"))
        cell_text.append(row_text)

    n_rows, n_cols = len(cell_text), len(col_labels_full)
    fig, ax = plt.subplots(figsize=(max(8, 2 + n_cols * 2.8), max(2, 0.5 + n_rows * 0.5)))
    ax.axis("off")
    tbl = ax.table(cellText=cell_text, rowLabels=row_labels, colLabels=col_labels_full,
                    cellLoc="center", rowLoc="right", loc="center")
    tbl.auto_set_font_size(False); tbl.set_fontsize(9); tbl.scale(1, 1.7)
    for j in range(n_cols):
        tbl[(0, j)].set_facecolor("#CCCCCC"); tbl[(0, j)].set_text_props(fontweight="bold")
    for i in range(n_rows):
        tbl[(i + 1, -1)].set_facecolor("#F5F5F5")
    if title:
        ax.set_title(title, pad=12, fontsize=11, fontweight="bold")
    plt.tight_layout()
    save(fig, path, cfg)


def render_dist_diff_table(pairs_results, path, cfg, title=""):
    rows = [[f"{r['pair'][0]} vs {r['pair'][1]}", f"{r['ks_stat']:.3f}",
             f"{r['ks_p']:.3f}" if r['ks_p'] >= 0.001 else "<0.001",
             f"{r['wasserstein']:.3f}", f"{r['kl']:.3f}", str(r['n1']), str(r['n2'])]
            for r in pairs_results if r is not None]
    if not rows:
        return
    col_labels = ["Pair", "KS stat", "KS p", "Wasserstein", "KL div", "n₁", "n₂"]
    fig, ax = plt.subplots(figsize=(max(10, 2 + len(col_labels) * 1.8), max(2, 0.5 + len(rows) * 0.45)))
    ax.axis("off")
    tbl = ax.table(cellText=rows, colLabels=col_labels, cellLoc="center", loc="center")
    tbl.auto_set_font_size(False); tbl.set_fontsize(9); tbl.scale(1, 1.6)
    for j in range(len(col_labels)):
        tbl[(0, j)].set_facecolor("#CCCCCC"); tbl[(0, j)].set_text_props(fontweight="bold")
    if title:
        ax.set_title(title, pad=12, fontsize=11, fontweight="bold")
    plt.tight_layout()
    save(fig, path, cfg)


def render_simple_table_png(rows, col_labels, path, cfg, title=""):
    if not rows:
        return
    fig, ax = plt.subplots(figsize=(max(8, 2 + len(col_labels) * 2.2), max(2, 0.5 + len(rows) * 0.45)))
    ax.axis("off")
    tbl = ax.table(cellText=rows, colLabels=col_labels, cellLoc="center", loc="center")
    tbl.auto_set_font_size(False); tbl.set_fontsize(9); tbl.scale(1, 1.6)
    for j in range(len(col_labels)):
        tbl[(0, j)].set_facecolor("#CCCCCC"); tbl[(0, j)].set_text_props(fontweight="bold")
    if title:
        ax.set_title(title, pad=12, fontsize=11, fontweight="bold")
    plt.tight_layout()
    save(fig, path, cfg)


def _overlaid_density_chart(groups, out_path, cfg, title="", xlabel=""):
    """groups: lista de (label, valores, chave_de_cor). Sem formas circulares/3D."""
    pal, ht, ls = _palette(cfg), _hatches(cfg), _linestyles(cfg)
    fig, ax = plt.subplots(figsize=cfg["figures"]["figsize"])
    for label, vals, key in groups:
        vals = np.asarray(vals, dtype=float)
        vals = vals[~np.isnan(vals)]
        if len(vals) == 0:
            continue
        color, hatch, style = pal.get(key, "#888888"), ht.get(key, ""), ls.get(key, "solid")
        ax.hist(vals, bins=30, density=True, histtype="step", linewidth=2.2,
                linestyle=style, color=color, label=f"{label} (n={len(vals)})")
        ax.hist(vals, bins=30, density=True, alpha=0.15, color=color, hatch=hatch)
    ax.set_xlabel(xlabel); ax.set_ylabel("Densidade"); ax.set_title(title)
    ax.legend(); ax.grid(cfg["figures"]["grid"], alpha=0.3)
    save(fig, out_path, cfg)


def _score_diff_bars(df_orig, df_1b, df_7b, cfg, out_dir, prefix="exp1", title="Score Difference"):
    """Usado por Exp1 e Exp2 — barras de diferença de score (Original − Gerado) por atributo."""
    attrs = cfg["score_attributes"]
    pal, ht = _palette(cfg), _hatches(cfg)
    if df_orig is not None:
        attrs = [a for a in attrs if a in df_orig.columns]

    def align(df_a, df_b, attr):
        if df_a is None or df_b is None:
            return None, None
        a, b = df_a.copy(), df_b.copy()
        if "stem" not in a.columns: a["stem"] = a["filename"].apply(_stem)
        if "stem" not in b.columns: b["stem"] = b["filename"].apply(_stem)
        m = a[["stem", attr]].merge(b[["stem", attr]].rename(columns={attr: attr + "_b"}), on="stem").dropna()
        if len(m) == 0:
            return None, None
        return m[attr].values, m[attr + "_b"].values

    diffs_1b, diffs_7b, attr_labels = [], [], []
    for attr in attrs:
        a_vals, b1_vals = align(df_orig, df_1b, attr)
        _, b7_vals = align(df_orig, df_7b, attr)
        diffs_1b.append(float(np.mean(a_vals - b1_vals)) if a_vals is not None and b1_vals is not None else None)
        diffs_7b.append(float(np.mean(a_vals - b7_vals)) if a_vals is not None and b7_vals is not None else None)
        attr_labels.append(attr_label(cfg, attr))

    valid_idx = [i for i in range(len(attrs)) if diffs_1b[i] is not None or diffs_7b[i] is not None]
    if not valid_idx:
        return
    x = np.arange(len(valid_idx)); width = 0.35
    fig, ax = plt.subplots(figsize=(max(10, len(valid_idx) * 0.9), 6))
    labels_v = [attr_labels[i] for i in valid_idx]
    d1 = [diffs_1b[i] if diffs_1b[i] is not None else 0 for i in valid_idx]
    d7 = [diffs_7b[i] if diffs_7b[i] is not None else 0 for i in valid_idx]
    ax.bar(x - width/2, d1, width, label="Orig − Janus-1B", color=pal["janus_1b"], hatch=ht["janus_1b"], edgecolor="black")
    ax.bar(x + width/2, d7, width, label="Orig − Janus-7B", color=pal["janus_7b"], hatch=ht["janus_7b"], edgecolor="black")
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_xticks(x); ax.set_xticklabels(labels_v, rotation=40, ha="right")
    ax.set_ylabel("Average Score Difference"); ax.set_title(title)
    ax.legend(); ax.grid(cfg["figures"]["grid"], alpha=0.3, axis="y")
    plt.tight_layout()
    save(fig, os.path.join(out_dir, f"{prefix}_score_diff_bars.png"), cfg)


print("OK — renderizadores definidos.")

## Exp 1 — APDDv2

Roda tudo de uma vez (mesmo conteúdo do `analyse_exp1` do script). Se quiser
mexer só num gráfico específico, copia o trecho pra uma célula nova — todas as
variáveis (`df_orig`, `df_1b`, `df_7b`, `attrs`...) continuam disponíveis
depois que a célula roda.

In [ ]:
def analyse_exp1(cfg, out_dir):
    exp_dir = _exp1_scores_dir(cfg)
    alpha = cfg["stats"]["alpha"]
    pal, ht = _palette(cfg), _hatches(cfg)

    df_orig = load_scores(exp_dir, "original")
    df_1b = load_scores(exp_dir, "Janus-Pro-1B")
    df_7b = load_scores(exp_dir, "Janus-Pro-7B")
    df_human = load_human_gt(cfg)

    if df_orig is None:
        print("[exp1] scores não encontrados, pulando.")
        return

    total_attr = "Total aesthetic score"
    attrs = _available_attrs(cfg, df_orig, df_1b, df_7b)

    # ── 1. Distribuição de scores ───────────────────────────────────────
    fig, ax = plt.subplots(figsize=cfg["figures"]["figsize"])
    for src, df, key in [("original", df_orig, "original"), ("Janus-Pro-1B", df_1b, "janus_1b"),
                         ("Janus-Pro-7B", df_7b, "janus_7b")]:
        if df is None or total_attr not in df.columns:
            continue
        ax.hist(df[total_attr].dropna(), bins=30, alpha=0.6, color=pal[key],
                label=ds_label(cfg, key), hatch=ht[key], edgecolor="black")
    ax.set_xlabel(L(cfg, "axes", "score")); ax.set_ylabel("Count")
    ax.set_title(L(cfg, "titles", "dist_scores")); ax.legend()
    ax.grid(cfg["figures"]["grid"], alpha=0.3)
    save(fig, os.path.join(out_dir, "exp1_score_distributions.png"), cfg)

    # ── 2. Boxplot por fonte ─────────────────────────────────────────────
    sources = [("original", df_orig, "original"), ("Janus-Pro-1B", df_1b, "janus_1b"),
               ("Janus-Pro-7B", df_7b, "janus_7b")]
    available = [(n, d, k) for n, d, k in sources if d is not None and total_attr in d.columns]
    if available:
        fig, ax = plt.subplots(figsize=cfg["figures"]["figsize"])
        bp = ax.boxplot([d[total_attr].dropna().values for _, d, _ in available],
                        tick_labels=[ds_label(cfg, k) for _, _, k in available], patch_artist=True)
        _style_median(bp)
        for patch, (_, _, k) in zip(bp["boxes"], available):
            patch.set_facecolor(pal[k]); patch.set_hatch(ht[k])
        ax.set_ylabel(L(cfg, "axes", "score")); ax.set_title("APDDv2 — Score by Source")
        ax.grid(cfg["figures"]["grid"], alpha=0.3)
        save(fig, os.path.join(out_dir, "exp1_boxplot_sources.png"), cfg)

    # ── 3. Tabela por categoria ──────────────────────────────────────────
    if "category" in df_orig.columns:
        cats = df_orig["category"].dropna().unique()
        rows, row_labels = [], []
        for cat in sorted(cats):
            sub = df_orig[df_orig["category"] == cat][total_attr].dropna()
            if len(sub) == 0:
                continue
            rows.append([f"{sub.mean():.2f}", f"{sub.std():.2f}", str(len(sub))])
            row_labels.append(str(cat))
        if rows:
            fig, ax = plt.subplots(figsize=(8, max(3, 0.4 * len(rows) + 1)))
            ax.axis("off")
            tbl = ax.table(cellText=rows, rowLabels=row_labels, colLabels=["Mean Score", "Std", "N"],
                           cellLoc="center", rowLoc="right", loc="center")
            tbl.auto_set_font_size(False); tbl.set_fontsize(9); tbl.scale(1, 1.6)
            for j in range(3):
                tbl[(0, j)].set_facecolor("#CCCCCC"); tbl[(0, j)].set_text_props(fontweight="bold")
            ax.set_title("APDDv2 — Score by Category", pad=12, fontsize=11, fontweight="bold")
            plt.tight_layout()
            save(fig, os.path.join(out_dir, "exp1_by_category_table.png"), cfg)

    # ── 4. Tabela estatística (Friedman + Wilcoxon + CLD) ────────────────
    groups, group_order = {}, []
    for gname, df in [("original", df_orig), ("Janus-Pro-1B", df_1b), ("Janus-Pro-7B", df_7b)]:
        if df is not None:
            d = df.copy()
            if "stem" not in d.columns: d["stem"] = d["filename"].apply(_stem)
            groups[gname] = d; group_order.append(gname)
    if len(groups) >= 2:
        fw = friedman_wilcoxon(groups, attrs, alpha)
        render_stat_table_png(fw, attrs, group_order, os.path.join(out_dir, "exp1_stat_table.png"),
                              cfg, title="APDDv2 — Friedman + Wilcoxon (CLD)")

    # ── 5. Barras de diferença de score ──────────────────────────────────
    if df_1b is not None or df_7b is not None:
        _score_diff_bars(df_orig, df_1b, df_7b, cfg, out_dir, prefix="exp1",
                         title="APDDv2 — Score Difference (Original − Generated)")

    # ── 6. Clusters (PCA + KMeans) ────────────────────────────────────────
    n_clusters, n_top = cfg["clustering"]["n_clusters"], cfg["clustering"]["n_top"]
    sub = df_orig[attrs].dropna()
    if len(sub) >= n_clusters * 2:
        X = StandardScaler().fit_transform(sub.values)
        km = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        labels = km.fit_predict(X)
        X2 = PCA(n_components=2, random_state=42).fit_transform(X)
        cluster_colors = ["#448FF2", "#33A650", "#F2A007"][:n_clusters]
        total_scores = df_orig.loc[sub.index, total_attr].values
        top_idx, bot_idx = np.argsort(total_scores)[-n_top:], np.argsort(total_scores)[:n_top]

        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        for ax_idx, (suffix, highlight_idx) in enumerate([(f"Top-{n_top}", top_idx), (f"Bottom-{n_top}", bot_idx)]):
            ax = axes[ax_idx]
            for c in range(n_clusters):
                mask = labels == c
                ax.scatter(X2[mask, 0], X2[mask, 1], alpha=0.3, s=20, color=cluster_colors[c], label=f"Cluster {c+1}")
            for i in highlight_idx:
                c = labels[i]
                ax.scatter(X2[i, 0], X2[i, 1], s=100, color=cluster_colors[c], edgecolors="black", linewidths=1.5, zorder=5)
            ax.set_title(f"PCA Clusters — {suffix}"); ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
            ax.legend(fontsize=8); ax.grid(cfg["figures"]["grid"], alpha=0.3)
        plt.suptitle("APDDv2 — Cluster Analysis", fontsize=14, fontweight="bold")
        plt.tight_layout()
        save(fig, os.path.join(out_dir, "exp1_clusters.png"), cfg)

        fig, ax = plt.subplots(figsize=(max(10, len(attrs) * 0.9), 6))
        x = np.arange(len(attrs)); width = 0.8 / n_clusters
        for c in range(n_clusters):
            mask = labels == c
            means = [float(sub.iloc[mask][a].mean()) for a in attrs]
            ax.bar(x + c * width, means, width, label=f"Cluster {c+1}", color=cluster_colors[c], edgecolor="black", alpha=0.85)
        ax.set_xticks(x + width * (n_clusters - 1) / 2)
        ax.set_xticklabels([attr_label(cfg, a) for a in attrs], rotation=40, ha="right")
        ax.set_ylabel(L(cfg, "axes", "score")); ax.set_title("Mean Attributes per Cluster")
        ax.legend(); ax.grid(cfg["figures"]["grid"], alpha=0.3, axis="y")
        plt.tight_layout()
        save(fig, os.path.join(out_dir, "exp1_cluster_attrs.png"), cfg)

    # ── 7. Diferenças de distribuição vs. Human GT ───────────────────────
    if df_human is not None:
        pairs = []
        for gname, df in [("Original", df_orig), ("Janus-1B", df_1b), ("Janus-7B", df_7b)]:
            if df is None:
                continue
            r, t = apply_nan_mask(df_human, df, total_attr)
            res = distribution_diff(r.dropna(), t.dropna(), "Human GT", gname)
            if res:
                pairs.append(res)
        if pairs:
            render_dist_diff_table(pairs, os.path.join(out_dir, "exp1_dist_diff.png"), cfg,
                                   title=f"APDDv2 — Distribution Differences ({attr_label(cfg, total_attr)})")

        all_pairs = []
        for attr in attrs:
            for gname, df in [("Janus-1B", df_1b), ("Janus-7B", df_7b)]:
                if df is None:
                    continue
                r, t = apply_nan_mask(df_human, df, attr)
                res = distribution_diff(r.dropna(), t.dropna(), f"Human/{attr_label(cfg, attr)}", gname)
                if res:
                    all_pairs.append(res)
        if all_pairs:
            render_dist_diff_table(all_pairs, os.path.join(out_dir, "exp1_dist_diff_full.png"), cfg,
                                   title="APDDv2 — Distribution Differences (all attributes)")


analyse_exp1(cfg, FIG_DIR)

## Exp 2 — Portinari

In [ ]:
def analyse_exp2(cfg, out_dir):
    pal, ht = _palette(cfg), _hatches(cfg)
    attrs = cfg["score_attributes"]
    alpha = cfg["stats"]["alpha"]
    total_attr = "Total aesthetic score"

    def _load_exp2(name):
        d = os.path.join(cfg["paths"]["outputs"], name)
        return load_scores(d, "original"), load_scores(d, "Janus-Pro-1B"), load_scores(d, "Janus-Pro-7B")

    df_2a_o, df_2a_1b, df_2a_7b = _load_exp2("exp2a_portinari")
    df_2b_o, df_2b_1b, df_2b_7b = _load_exp2("exp2b_portinari_human")

    def _total(df):
        return df[total_attr].dropna().values if df is not None and total_attr in df.columns else None

    sources_box = [
        ("Human_description", df_2b_o, "original"), ("Gen_description", df_2a_o, "janus_1b"),
        ("Janus-Pro-1B (2a)", df_2a_1b, "janus_1b"), ("Janus-Pro-7B (2a)", df_2a_7b, "janus_7b"),
        ("Janus-Pro-1B (2b)", df_2b_1b, "janus_1b"), ("Janus-Pro-7B (2b)", df_2b_7b, "janus_7b"),
    ]
    available_box = [(n, d, k) for n, d, k in sources_box if _total(d) is not None]
    if available_box:
        fig, ax = plt.subplots(figsize=(max(10, len(available_box) * 1.5), 6))
        bp = ax.boxplot([_total(d) for _, d, _ in available_box], tick_labels=[n for n, _, _ in available_box],
                        patch_artist=True)
        _style_median(bp)
        for patch, (_, _, k) in zip(bp["boxes"], available_box):
            patch.set_facecolor(pal[k]); patch.set_hatch(ht[k])
        ax.set_ylabel(L(cfg, "axes", "score")); ax.set_title("Portinari — Score by Source")
        ax.grid(cfg["figures"]["grid"], alpha=0.3)
        plt.xticks(rotation=25, ha="right"); plt.tight_layout()
        save(fig, os.path.join(out_dir, "exp2_boxplot.png"), cfg)

    for exp_tag, df_o, df_1b, df_7b in [("2a_gen_captions", df_2a_o, df_2a_1b, df_2a_7b),
                                         ("2b_human_captions", df_2b_o, df_2b_1b, df_2b_7b)]:
        groups, group_order = {}, []
        for gname, df in [("original", df_o), ("Janus-Pro-1B", df_1b), ("Janus-Pro-7B", df_7b)]:
            if df is not None:
                d = df.copy()
                if "stem" not in d.columns: d["stem"] = d["filename"].apply(_stem)
                groups[gname] = d; group_order.append(gname)
        if len(groups) >= 2:
            fw = friedman_wilcoxon(groups, attrs, alpha)
            render_stat_table_png(fw, attrs, group_order, os.path.join(out_dir, f"exp2_{exp_tag}_stat_table.png"),
                                  cfg, title=f"Portinari ({exp_tag}) — Friedman + Wilcoxon")

    if df_2a_o is not None:
        _score_diff_bars(df_2a_o, df_2a_1b, df_2a_7b, cfg, out_dir, prefix="exp2a",
                         title="Portinari (AI Captions) — Score Difference")
    if df_2b_o is not None:
        _score_diff_bars(df_2b_o, df_2b_1b, df_2b_7b, cfg, out_dir, prefix="exp2b",
                         title="Portinari (Human Captions) — Score Difference")

    df_human = load_human_gt(cfg)
    all_pairs = []
    pair_defs = [("Portinari orig (2a)", df_2a_o), ("Portinari orig (2b)", df_2b_o),
                 ("Janus-1B (2a)", df_2a_1b), ("Janus-7B (2a)", df_2a_7b),
                 ("Janus-1B (2b)", df_2b_1b), ("Janus-7B (2b)", df_2b_7b)]
    if df_human is not None:
        for name, df in pair_defs:
            if df is None or total_attr not in df.columns:
                continue
            res = distribution_diff(df_human[total_attr].dropna() if total_attr in df_human.columns else pd.Series([]),
                                    df[total_attr].dropna(), "APDDv2-Human", name)
            if res:
                all_pairs.append(res)
    for n1, d1 in pair_defs:
        for n2, d2 in pair_defs:
            if n1 >= n2 or d1 is None or d2 is None:
                continue
            if total_attr not in d1.columns or total_attr not in d2.columns:
                continue
            res = distribution_diff(d1[total_attr].dropna(), d2[total_attr].dropna(), n1, n2)
            if res:
                all_pairs.append(res)
    if all_pairs:
        render_dist_diff_table(all_pairs, os.path.join(out_dir, "exp2_dist_diff.png"), cfg,
                               title="Portinari — Distribution Differences (Total Score)")


analyse_exp2(cfg, FIG_DIR)

## Exp 3 — MNIST

In [ ]:
def analyse_exp3(cfg, out_dir):
    pal, ht = _palette(cfg), _hatches(cfg)
    total_attr = "Total aesthetic score"

    df_mnist = load_scores(os.path.join(cfg["paths"]["outputs"], "exp3_mnist"), "original")
    df_apdd = load_scores(_exp1_scores_dir(cfg), "original")
    df_port = load_scores(os.path.join(cfg["paths"]["outputs"], "exp2a_portinari"), "original")

    if df_mnist is None or total_attr not in df_mnist.columns:
        print("[exp3] scores não encontrados, pulando.")
        return

    sources = []
    if df_apdd is not None and total_attr in df_apdd.columns:
        sources.append(("APDDv2", df_apdd[total_attr].dropna(), "original"))
    if df_port is not None and total_attr in df_port.columns:
        sources.append(("Portinari", df_port[total_attr].dropna(), "original"))
    sources.append(("MNIST", df_mnist[total_attr].dropna(), "mnist"))

    if len(sources) >= 2:
        fig, ax = plt.subplots(figsize=(8, 6))
        bp = ax.boxplot([s[1].values for s in sources], tick_labels=[s[0] for s in sources], patch_artist=True)
        _style_median(bp)
        for patch, (_, _, k) in zip(bp["boxes"], sources):
            patch.set_facecolor(pal[k]); patch.set_hatch(ht[k])
        ax.set_ylabel(L(cfg, "axes", "score")); ax.set_title(L(cfg, "titles", "art_vs_noart"))
        ax.grid(cfg["figures"]["grid"], alpha=0.3)
        save(fig, os.path.join(out_dir, "exp3_art_vs_noart.png"), cfg)

    pairs = []
    mnist_s = df_mnist[total_attr].dropna()
    if df_apdd is not None and total_attr in df_apdd.columns:
        pairs.append(distribution_diff(df_apdd[total_attr].dropna(), mnist_s, "APDDv2", "MNIST"))
    if df_port is not None and total_attr in df_port.columns:
        pairs.append(distribution_diff(df_port[total_attr].dropna(), mnist_s, "Portinari", "MNIST"))
    pairs = [p for p in pairs if p is not None]
    if pairs:
        render_dist_diff_table(pairs, os.path.join(out_dir, "exp3_dist_diff.png"), cfg,
                               title="Art vs Non-Art — Distribution Differences")


analyse_exp3(cfg, FIG_DIR)

## Comparações — Fig 4.9 (APDDv2 vs Portinari) e Fig 4.10 (Art vs Non-Art)

In [ ]:
def analyse_comparisons(cfg, out_dir):
    pal, ht = _palette(cfg), _hatches(cfg)
    attrs = cfg["score_attributes"]

    df_apdd = load_scores(_exp1_scores_dir(cfg), "original")
    df_port = load_scores(os.path.join(cfg["paths"]["outputs"], "exp2a_portinari"), "original")
    df_mnist = load_scores(os.path.join(cfg["paths"]["outputs"], "exp3_mnist"), "original")

    if df_apdd is not None and df_port is not None:
        valid_attrs = [a for a in attrs if a in df_apdd.columns and a in df_port.columns]
        if valid_attrs:
            apdd_means = [df_apdd[a].dropna().mean() for a in valid_attrs]
            port_means = [df_port[a].dropna().mean() for a in valid_attrs]
            x = np.arange(len(valid_attrs)); width = 0.35
            fig, ax = plt.subplots(figsize=(max(10, len(valid_attrs) * 0.9), 6))
            ax.bar(x - width/2, apdd_means, width, label="APDDv2", color=pal["original"], hatch=ht["original"], edgecolor="black")
            ax.bar(x + width/2, port_means, width, label="Portinari", color=pal["janus_1b"], hatch=ht["janus_1b"], edgecolor="black")
            ax.set_xticks(x); ax.set_xticklabels([attr_label(cfg, a) for a in valid_attrs], rotation=40, ha="right")
            ax.set_ylabel(L(cfg, "axes", "score")); ax.set_title("Fig 4.9 — APDDv2 vs Portinari: Mean Scores per Attribute")
            ax.legend(); ax.grid(cfg["figures"]["grid"], alpha=0.3, axis="y")
            plt.tight_layout()
            save(fig, os.path.join(out_dir, "fig49_apddv2_vs_portinari.png"), cfg)

    if df_mnist is not None:
        art_dfs = [(d, n, k) for d, n, k in [(df_apdd, "APDDv2", "original"), (df_port, "Portinari", "original")] if d is not None]
        valid_attrs = [a for a in attrs if all(a in d.columns for d, _, _ in art_dfs) and a in df_mnist.columns]
        if valid_attrs and art_dfs:
            x = np.arange(len(valid_attrs)); width = 0.8 / (len(art_dfs) + 1)
            fig, ax = plt.subplots(figsize=(max(10, len(valid_attrs) * 0.9), 6))
            for i, (d, name, key) in enumerate(art_dfs):
                means = [d[a].dropna().mean() for a in valid_attrs]
                ax.bar(x + i * width, means, width, label=name, color=pal[key], hatch=ht[key], edgecolor="black", alpha=0.85)
            mnist_means = [df_mnist[a].dropna().mean() for a in valid_attrs]
            ax.bar(x + len(art_dfs) * width, mnist_means, width, label="MNIST", color=pal["mnist"], hatch=ht["mnist"], edgecolor="black", alpha=0.85)
            ax.set_xticks(x + width * len(art_dfs) / 2)
            ax.set_xticklabels([attr_label(cfg, a) for a in valid_attrs], rotation=40, ha="right")
            ax.set_ylabel(L(cfg, "axes", "score")); ax.set_title("Fig 4.10 — Art vs Non-Art: Mean Scores per Attribute")
            ax.legend(); ax.grid(cfg["figures"]["grid"], alpha=0.3, axis="y")
            plt.tight_layout()
            save(fig, os.path.join(out_dir, "fig410_art_vs_noart.png"), cfg)


analyse_comparisons(cfg, FIG_DIR)

## Exp 4 — Ruído

In [ ]:
def analyse_exp4(cfg, out_dir):
    df = load_scores(os.path.join(cfg["paths"]["outputs"], "exp4_noise"), "original")
    if df is None:
        print("[exp4] scores não encontrados, pulando."); return
    if "noise_type" not in df.columns:
        print("[exp4] coluna noise_type ausente no CSV — reprocesse o scoring."); return

    total_attr = "Total aesthetic score"
    noise_types = df["noise_type"].dropna().unique()
    noise_colors = {"gaussian": "#448FF2", "blur": "#33A650", "shapes": "#F2A007"}
    noise_hatches = {"gaussian": "///", "blur": "xxx", "shapes": "..."}
    noise_ls = {"gaussian": "solid", "blur": "dashed", "shapes": "dotted"}
    noise_mk = {"gaussian": "o", "blur": "s", "shapes": "^"}

    fig, ax = plt.subplots(figsize=cfg["figures"]["figsize"])
    for nt in sorted(noise_types):
        sub = df[df["noise_type"] == nt]
        if "noise_level" not in sub.columns or total_attr not in sub.columns:
            continue
        grp = sub.groupby("noise_level")[total_attr]
        levels = sorted(grp.groups.keys())
        means = np.array([grp.get_group(l).mean() for l in levels], dtype=float)
        sems = np.nan_to_num(np.array([grp.get_group(l).sem() for l in levels], dtype=float), 0)
        label = L(cfg, "noise_types", nt) if nt in cfg["labels"][cfg["lang"]].get("noise_types", {}) else nt
        color = noise_colors.get(nt, "#888888")
        ax.plot(levels, means, color=color, linestyle=noise_ls.get(nt, "solid"), marker=noise_mk.get(nt, "o"), label=label)
        ax.fill_between(levels, means - sems, means + sems, alpha=0.15, color=color)
    ax.set_xlabel(L(cfg, "axes", "noise_level")); ax.set_ylabel(L(cfg, "axes", "score"))
    ax.set_title(L(cfg, "titles", "noise_impact")); ax.legend(); ax.grid(cfg["figures"]["grid"], alpha=0.3)
    save(fig, os.path.join(out_dir, "exp4_noise_impact.png"), cfg)

    if total_attr in df.columns:
        present = [nt for nt in sorted(noise_types) if nt in df["noise_type"].values]
        box_data = [df[df["noise_type"] == nt][total_attr].dropna().values for nt in present]
        box_labels = [L(cfg, "noise_types", nt) if nt in cfg["labels"][cfg["lang"]].get("noise_types", {}) else nt for nt in present]
        if box_data:
            fig, ax = plt.subplots(figsize=(8, 6))
            bp = ax.boxplot(box_data, tick_labels=box_labels, patch_artist=True)
            _style_median(bp)
            for patch, nt in zip(bp["boxes"], present):
                patch.set_facecolor(noise_colors.get(nt, "#888888")); patch.set_hatch(noise_hatches.get(nt, ""))
            ax.set_ylabel(L(cfg, "axes", "score")); ax.set_title("Noise Types — Score Distribution")
            ax.grid(cfg["figures"]["grid"], alpha=0.3)
            save(fig, os.path.join(out_dir, "exp4_noise_boxplot.png"), cfg)

    df_base = load_scores(_exp1_scores_dir(cfg), "original")
    if df_base is not None and total_attr in df_base.columns:
        pairs = []
        for nt in sorted(noise_types):
            sub = df[df["noise_type"] == nt][total_attr].dropna()
            if len(sub) < 2:
                continue
            label_nt = L(cfg, "noise_types", nt) if nt in cfg["labels"][cfg["lang"]].get("noise_types", {}) else nt
            res = distribution_diff(df_base[total_attr].dropna(), sub, "APDDv2 Original", label_nt)
            if res:
                pairs.append(res)
        if pairs:
            render_dist_diff_table(pairs, os.path.join(out_dir, "exp4_dist_diff.png"), cfg,
                                   title="Noise — Distribution Differences vs APDDv2 Baseline")


analyse_exp4(cfg, FIG_DIR)

## Exp 5 — Temporal

In [ ]:
def analyse_exp5(cfg, out_dir):
    pal = _palette(cfg)
    total_attr = "Total aesthetic score"
    alpha = cfg["stats"]["alpha"]

    df5a = load_scores(os.path.join(cfg["paths"]["outputs"], "exp5a_temporal"), "original")
    df5b = load_scores(os.path.join(cfg["paths"]["outputs"], "exp5b_temporal_error"), "original")

    if df5a is not None and "frame_idx" in df5a.columns and total_attr in df5a.columns:
        fig, ax = plt.subplots(figsize=cfg["figures"]["figsize"])
        grp = df5a.groupby("frame_idx")[total_attr]
        frames = sorted(grp.groups.keys())
        means = np.array([grp.get_group(f).mean() for f in frames])
        sems = np.nan_to_num(np.array([grp.get_group(f).sem() for f in frames], dtype=float), 0)
        ax.plot(frames, means, color=pal["original"], linewidth=2, marker="o", markersize=4, label="Mean score")
        ax.fill_between(frames, means - sems, means + sems, alpha=0.2, color=pal["original"])
        ax.set_xlabel(L(cfg, "axes", "frame_idx")); ax.set_ylabel(L(cfg, "axes", "score"))
        ax.set_title(L(cfg, "titles", "temporal_consist")); ax.legend(); ax.grid(cfg["figures"]["grid"], alpha=0.3)
        save(fig, os.path.join(out_dir, "exp5a_temporal_consistency.png"), cfg)

    if df5b is not None and total_attr in df5b.columns:
        if "frame_idx" in df5b.columns:
            fig, ax = plt.subplots(figsize=cfg["figures"]["figsize"])
            grp = df5b.groupby("frame_idx")[total_attr]
            frames = sorted(grp.groups.keys())
            means = np.array([float(grp.get_group(f).mean()) for f in frames])
            sems = np.nan_to_num(np.array([grp.get_group(f).sem() for f in frames], dtype=float), 0)
            colors = ["#F23838" if f >= 12 else pal["original"] for f in frames]
            ax.plot(frames, means, color=pal["original"], linewidth=1.5, zorder=1)
            ax.scatter(frames, means, c=colors, s=40, zorder=2)
            ax.fill_between(frames, means - sems, means + sems, alpha=0.15, color=pal["original"])
            ax.axvline(12, color="#F23838", linestyle="--", linewidth=1.5, label="Error starts (frame 12)")
            ax.set_xlabel(L(cfg, "axes", "frame_idx")); ax.set_ylabel(L(cfg, "axes", "score"))
            ax.set_title("Exp5b — Score Drop at Error Frame"); ax.legend(); ax.grid(cfg["figures"]["grid"], alpha=0.3)
            save(fig, os.path.join(out_dir, "exp5b_frame_score.png"), cfg)

        if "degradation_pct" in df5b.columns:
            grp_deg = df5b.groupby("degradation_pct")[total_attr]
            levels = sorted(grp_deg.groups.keys())
            if len(levels) >= 2:
                base_scores = grp_deg.get_group(levels[0]).dropna().values
                means, pvals = [], []
                n_tests = len(levels) - 1
                for lvl in levels:
                    sub = grp_deg.get_group(lvl).dropna().values
                    means.append(float(np.mean(sub)))
                    if lvl == levels[0] or len(sub) < 2 or len(base_scores) < 2:
                        pvals.append(1.0)
                    else:
                        try:
                            _, p = mannwhitneyu(base_scores, sub, alternative="two-sided")
                            pvals.append(float(p))
                        except Exception:
                            pvals.append(1.0)
                threshold_lvl = next((lvl for lvl, p in zip(levels[1:], pvals[1:]) if p < (alpha / n_tests)), None)
                fig, ax = plt.subplots(figsize=cfg["figures"]["figsize"])
                ax.plot(levels, means, color=pal["original"], linewidth=2, marker="o")
                if threshold_lvl is not None:
                    ax.axvline(threshold_lvl, color=pal["highlight"], linestyle="--", linewidth=2,
                               label=f"Significance threshold ({threshold_lvl}%)")
                    ax.legend()
                ax.set_xlabel(L(cfg, "axes", "degradation")); ax.set_ylabel(L(cfg, "axes", "score"))
                ax.set_title(L(cfg, "titles", "degradation_detect")); ax.grid(cfg["figures"]["grid"], alpha=0.3)
                save(fig, os.path.join(out_dir, "exp5b_degradation.png"), cfg)


analyse_exp5(cfg, FIG_DIR)

## Diagnósticos avançados

Sem outro modelo pra comparar — validam o ArtCLIP contra si mesmo:
1. **Monotonicidade** — score cai conforme o ruído aumenta? (Exp4)
2. **Validade discriminante** — separa Humano / Sintético / MNIST?
3. **Viés cultural** — score difere dentro vs. fora da base de treino (Exp1 vs Exp2)?
4. **Grupos de dificuldade** — Fácil/Médio/Difícil, regra de monotonicidade + erro de calibração

In [ ]:
def analyse_monotonicity(cfg, out_dir):
    df = load_scores(os.path.join(cfg["paths"]["outputs"], "exp4_noise"), "original")
    total_attr = "Total aesthetic score"
    if df is None or "noise_type" not in df.columns or "noise_level" not in df.columns:
        print("[monotonicity] scores/metadados do exp4 não encontrados, pulando."); return

    alpha = cfg["stats"]["alpha"]
    rows = []
    for nt in sorted(df["noise_type"].dropna().unique()):
        sub = df[df["noise_type"] == nt][["noise_level", total_attr]].dropna()
        if len(sub) < 3:
            continue
        rho, p_s = spearmanr(sub["noise_level"], sub[total_attr])
        tau, p_k = kendalltau(sub["noise_level"], sub[total_attr])
        if p_s < alpha and rho < 0:
            verdict = "Monotônico (decrescente)"
        elif p_s < alpha and rho > 0:
            verdict = "Anômalo (crescente)"
        else:
            verdict = "Não monotônico"
        label_nt = L(cfg, "noise_types", nt) if nt in cfg["labels"][cfg["lang"]].get("noise_types", {}) else nt
        rows.append([label_nt, f"{rho:.3f}", f"{p_s:.3f}" if p_s >= 0.001 else "<0.001",
                     f"{tau:.3f}", f"{p_k:.3f}" if p_k >= 0.001 else "<0.001", str(len(sub)), verdict])
    if rows:
        render_simple_table_png(rows, ["Tipo de ruído", "Spearman ρ", "p (ρ)", "Kendall τ", "p (τ)", "n", "Diagnóstico"],
                                os.path.join(out_dir, "monotonicity_table.png"), cfg,
                                title="Teste de Monotonicidade — Score vs. Nível de Ruído (Exp4)")


analyse_monotonicity(cfg, FIG_DIR)

In [ ]:
def analyse_discriminative_validity(cfg, out_dir):
    total_attr = "Total aesthetic score"
    exp1 = _exp1_scores_dir(cfg)
    df_human = load_scores(exp1, "original")
    df_1b = load_scores(exp1, "Janus-Pro-1B")
    df_7b = load_scores(exp1, "Janus-Pro-7B")
    df_mnist = load_scores(os.path.join(cfg["paths"]["outputs"], "exp3_mnist"), "original")

    groups = []
    for label, df, key in [("Humano (APDDv2)", df_human, "original"), ("Sintético (Janus-1B)", df_1b, "janus_1b"),
                           ("Sintético (Janus-7B)", df_7b, "janus_7b"), ("MNIST", df_mnist, "mnist")]:
        if df is not None and total_attr in df.columns:
            vals = df[total_attr].dropna().values
            if len(vals) > 0:
                groups.append((label, vals, key))
    if len(groups) < 2:
        print("[discriminative_validity] dados insuficientes, pulando."); return

    _overlaid_density_chart(groups, os.path.join(out_dir, "discriminative_validity_density.png"), cfg,
                            title="Validade Discriminante — Distribuição de Score por Grupo", xlabel=L(cfg, "axes", "score"))

    pairs = [distribution_diff(pd.Series(v1), pd.Series(v2), n1, n2)
             for (n1, v1, _), (n2, v2, _) in combinations(groups, 2)]
    pairs = [p for p in pairs if p]
    if pairs:
        render_dist_diff_table(pairs, os.path.join(out_dir, "discriminative_validity_table.png"), cfg,
                               title="Validade Discriminante — KS Test entre Grupos")


analyse_discriminative_validity(cfg, FIG_DIR)

In [ ]:
def analyse_cultural_bias(cfg, out_dir):
    total_attr = "Total aesthetic score"
    df_in = load_scores(_exp1_scores_dir(cfg), "original")
    df_out = load_scores(os.path.join(cfg["paths"]["outputs"], "exp2a_portinari"), "original")
    if df_in is None or df_out is None or total_attr not in df_in.columns or total_attr not in df_out.columns:
        print("[cultural_bias] scores não encontrados, pulando."); return

    vals_in, vals_out = df_in[total_attr].dropna(), df_out[total_attr].dropna()
    if len(vals_in) == 0 or len(vals_out) == 0:
        print("[cultural_bias] sem valores válidos, pulando."); return

    pal, ht = _palette(cfg), _hatches(cfg)
    fig, ax = plt.subplots(figsize=(7, 6))
    bp = ax.boxplot([vals_in.values, vals_out.values],
                    tick_labels=["APDDv2\n(dentro da base)", "Portinari\n(fora da base)"], patch_artist=True)
    _style_median(bp)
    for patch, key in zip(bp["boxes"], ["original", "janus_1b"]):
        patch.set_facecolor(pal[key]); patch.set_hatch(ht[key])
    ax.set_ylabel(L(cfg, "axes", "score")); ax.set_title("Viés Cultural — Score Dentro vs. Fora da Base de Treino")
    ax.grid(cfg["figures"]["grid"], alpha=0.3)
    save(fig, os.path.join(out_dir, "cultural_bias_boxplot.png"), cfg)

    res = distribution_diff(vals_in, vals_out, "APDDv2 (dentro)", "Portinari (fora)")
    if res:
        calibration_shift = float(vals_out.mean() - vals_in.mean())
        rows = [[f"{res['pair'][0]} vs {res['pair'][1]}", f"{vals_in.mean():.2f}", f"{vals_out.mean():.2f}",
                 f"{calibration_shift:+.2f}", f"{res['ks_stat']:.3f}",
                 f"{res['ks_p']:.3f}" if res['ks_p'] >= 0.001 else "<0.001", str(res['n1']), str(res['n2'])]]
        render_simple_table_png(rows, ["Comparação", "Média dentro", "Média fora", "Desvio de calibração",
                                       "KS stat", "KS p", "n₁", "n₂"],
                                os.path.join(out_dir, "cultural_bias_table.png"), cfg,
                                title="Viés Cultural — Desvio de Calibração (fora − dentro da base)")


analyse_cultural_bias(cfg, FIG_DIR)

In [ ]:
def analyse_difficulty_groups(cfg, out_dir):
    total_attr = "Total aesthetic score"
    exp1 = _exp1_scores_dir(cfg)
    df_easy = load_scores(exp1, "original")
    df_med = load_scores(exp1, "Janus-Pro-7B")

    df4 = load_scores(os.path.join(cfg["paths"]["outputs"], "exp4_noise"), "original")
    df_hard = None
    if df4 is not None and "noise_type" in df4.columns and "noise_level" in df4.columns:
        shapes = df4[df4["noise_type"] == "shapes"]
        if not shapes.empty:
            df_hard = shapes[shapes["noise_level"] == shapes["noise_level"].max()]

    groups = []
    for name, d, key in [("Fácil (Humano)", df_easy, "original"), ("Médio (Sintético limpo)", df_med, "janus_7b"),
                         ("Difícil (Sintético + ruído estrutural)", df_hard, "highlight")]:
        if d is None or total_attr not in d.columns:
            continue
        vals = d[total_attr].dropna().values
        if len(vals) > 0:
            groups.append((name, vals, key))
    if len(groups) < 2:
        print("[difficulty_groups] dados insuficientes, pulando."); return

    _overlaid_density_chart(groups, os.path.join(out_dir, "difficulty_groups_density.png"), cfg,
                            title="Matriz de Confusão Estética — Grupos de Dificuldade", xlabel=L(cfg, "axes", "score"))

    pal, ht = _palette(cfg), _hatches(cfg)
    fig, ax = plt.subplots(figsize=(8, 6))
    means = [float(np.mean(v)) for _, v, _ in groups]
    sems = [float(np.std(v, ddof=1) / np.sqrt(len(v))) if len(v) > 1 else 0.0 for _, v, _ in groups]
    x = np.arange(len(groups))
    bars = ax.bar(x, means, yerr=sems, capsize=5, color=[pal[k] for _, _, k in groups], edgecolor="black")
    for bar, (_, _, k) in zip(bars, groups):
        bar.set_hatch(ht[k])
    ax.set_xticks(x); ax.set_xticklabels([n for n, _, _ in groups], rotation=15, ha="right")
    ax.set_ylabel(L(cfg, "axes", "score"))
    monotonic = all(means[i] >= means[i + 1] for i in range(len(means) - 1))
    ax.set_title("Regra de Monotonicidade entre Grupos de Dificuldade\n"
                 + ("✓ Ordem preservada (Fácil > Médio > Difícil)" if monotonic else "✗ Ordem violada — possível erro de calibração"))
    ax.grid(cfg["figures"]["grid"], alpha=0.3, axis="y")
    plt.tight_layout()
    save(fig, os.path.join(out_dir, "difficulty_groups_means.png"), cfg)

    easy_vals = groups[0][1]
    easy_median = float(np.median(easy_vals)) if len(easy_vals) else float("nan")
    rows = []
    for name, vals, _ in groups:
        n_above = int(np.sum(vals > easy_median)) if not np.isnan(easy_median) else 0
        pct_above = 100 * n_above / len(vals) if len(vals) else 0.0
        rows.append([name, str(len(vals)), f"{np.mean(vals):.2f}", f"{np.median(vals):.2f}",
                     f"{np.std(vals):.2f}", f"{pct_above:.1f}%"])
    render_simple_table_png(rows, ["Grupo", "n", "Média", "Mediana", "Std", "% acima da mediana do Fácil"],
                            os.path.join(out_dir, "difficulty_groups_table.png"), cfg,
                            title=f"Grupos de Dificuldade — Estatísticas e Erros de Calibração (mediana Fácil={easy_median:.2f})")


analyse_difficulty_groups(cfg, FIG_DIR)

## Baixar os resultados

In [ ]:
import shutil
zip_path = shutil.make_archive("/content/figures_novas", "zip", FIG_DIR)
print("Zip criado:", zip_path)
files.download(zip_path)

## Como devolver pra mim

Quando terminar de ajustar as visualizações:
- **File → Download → Download .py** (vira um script) — me manda esse arquivo, ou
- **File → Download → Download .ipynb** — também funciona, eu extraio o código das células.

Me diz também **quais células você mudou** (ou só cola o trecho que alterou) pra eu
saber exatamente o que sincronizar de volta em `scripts/analyze.py` — não preciso
reprocessar o notebook inteiro, só as partes que você mexeu.